In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

In [2]:
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)

In [3]:
runs_folders = [
    "/home/bulatov/rmt/test-time/compressing-associations/runs",
    "/home/bulatov/rmt/test-time/compressing-associations/runs-rmca-versions",
]

In [4]:
base_dir = Path("/home/bulatov/rmt/test-time/compressing-associations")
all_runs = []
for runs_folder in runs_folders:
    runs_path = Path(runs_folder)
    target_runs = list(runs_path.glob('**/run_*'))
    
    for run_path in target_runs:
        if not run_path.is_dir():
            continue
        # skip checkpoint subdirs
        if 'checkpoint' in str(run_path):
            continue
            
        run_stats = {}
        run_stats['run_path'] = str(run_path)
        run_stats['runs_folder'] = runs_path.name
        run_stats['run_name'] = run_path.parent.name
        run_stats['task_name'] = run_path.parent.parent.name
        run_stats['run_id'] = int(run_path.name.split('_')[1])
        
        config_path = run_path / 'config.json'
        if config_path.exists():
            cli_args = json.load(open(config_path))['cli_args']
            run_stats.update(cli_args)
        
        results_path = run_path / 'all_results.json'
        if results_path.exists():
            results = json.load(open(results_path))
            run_stats.update(results)
        else:
            run_stats['eval_exact_match'] = np.nan
            run_stats['eval_token_accuracy'] = np.nan
        all_runs.append(run_stats)
df = pd.DataFrame(all_runs)
print(f"Total runs: {len(df)}")
print(f"Columns: {list(df.columns)}")
# df

Total runs: 450
Columns: ['run_path', 'runs_folder', 'run_name', 'task_name', 'run_id', 'exp_path', 'per_device_batch_size', 'data_path', 'tokenizer_path', 'gradient_accumulation_steps', 'total_batch_size', 'metric_for_best_model', 'warmup_steps', 'max_steps', 'logging_steps', 'eval_steps', 'weight_decay', 'learning_rate', 'lr_scheduler_type', 'early_stopping_patience', 'seed', 'base_model', 'n_layer', 'n_head', 'n_embd', 'n_mem_tokens', 'n_ctrl_tokens', 'use_mem_proj', 'mem_proj_mode', 'memory_task_freq', 'memory_task', 'memory_key_size', 'memory_value_size', 'model_cpt', 'pairs_per_segment', 'n_pairs', 'n_keys', 'n_values', 'epoch', 'eval_exact_match_None', 'eval_exact_match_base', 'eval_loss', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'eval_token_accuracy', 'patience', 'adam_beta1', 'adam_beta2', 'adam_epsilon', 'pretrained_model', 'max_position_embeddings', 'max_input_length', 'attn_implementation', 'flashrnn_backend', 'overwrite_output_dir', 'do_eval_only

In [5]:
import re

def parse_run_name(run_name):
    info = {}
    m = re.match(r'^(.+?)_L(\d+)H(\d+)D(\d+)(.*)', run_name)
    if not m:
        return info
    
    info['model'] = m.group(1)
    info['L'] = int(m.group(2))
    info['H'] = int(m.group(3))
    info['D'] = int(m.group(4))
    rest = m.group(5)
    
    mem_match = re.search(r'_mem(\d+)', rest)
    if mem_match:
        info['mem'] = int(mem_match.group(1))
    
    seg_match = re.search(r'-(\d+x\d+)', rest)
    if seg_match:
        info['segment'] = seg_match.group(1)
    
    return info

parsed = df['run_name'].apply(parse_run_name).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)

In [9]:
# df

In [10]:
df['eval_exact_match'] = df['eval_exact_match'].fillna(df['eval_exact_match_base'])
df = df.dropna(subset=['eval_exact_match'])

df.mem = df.mem.fillna(0).astype(int)
# df.segment = df.segment.fillna('1x' + df.

df['modification'] = ''
df.loc[df.max_steps == 200000, 'modification'] = '2x steps'

df['n_pairs'] = df.task_name.apply(lambda x: int(x[1:].split('-')[0]))

In [11]:
df[(df.task_name == 'N16-K2V2-V62_1M') & (df.base_model == 'mamba2')]

,run_path,runs_folder,run_name,task_name,run_id,exp_path,per_device_batch_size,data_path,tokenizer_path,gradient_accumulation_steps,total_batch_size,metric_for_best_model,warmup_steps,max_steps,logging_steps,eval_steps,weight_decay,learning_rate,lr_scheduler_type,early_stopping_patience,seed,base_model,n_layer,n_head,n_embd,n_mem_tokens,n_ctrl_tokens,use_mem_proj,mem_proj_mode,memory_task_freq,memory_task,memory_key_size,memory_value_size,model_cpt,pairs_per_segment,n_pairs,n_keys,n_values,epoch,eval_exact_match_None,eval_exact_match_base,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_token_accuracy,patience,adam_beta1,adam_beta2,adam_epsilon,pretrained_model,max_position_embeddings,max_input_length,attn_implementation,flashrnn_backend,overwrite_output_dir,do_eval_only,eval_exact_match,num_input_tokens_seen,model,L,H,D,mem,segment,modification
144,/home/bulatov/rmt/test-time/compressing-associ...,runs,mamba2_L4H4D128_bs_64_lr_1e-03_b2_0.99,N16-K2V2-V62_1M,1,./runs/N16-K2V2-V62_1M/mamba2_L4H4D128_bs_64_l...,64,./data/N16-K2V2-V62_1M,./tokenizers/kv_alphabet_62/,1,64,token_accuracy,10000,100000,500,500,0.0,0.0010,constant_with_warmup,50,143,mamba2,4,4,128,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,2.0,2.0,6.4,NaN,NaN,0.000911,1.7688,2826.806,44.664,0.9996,1.0,0.9,0.99,1.000000e-08,NaN,NaN,NaN,NaN,NaN,False,False,0.9992,819200000.0,mamba2,4,4,128,0,NaN,
148,/home/bulatov/rmt/test-time/compressing-associ...,runs,mamba2_L4H4D128_bs_64_lr_3e-04_b2_0.99,N16-K2V2-V62_1M,1,./runs/N16-K2V2-V62_1M/mamba2_L4H4D128_bs_64_l...,64,./data/N16-K2V2-V62_1M,./tokenizers/kv_alphabet_62/,1,64,token_accuracy,10000,100000,500,500,0.0,0.0003,constant_with_warmup,50,143,mamba2,4,4,128,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,2.0,2.0,6.4,NaN,NaN,0.001354,1.4662,3410.239,53.882,0.9992,1.0,0.9,0.99,1.000000e-08,NaN,NaN,NaN,NaN,NaN,False,False,0.9984,819200000.0,mamba2,4,4,128,0,NaN,


In [13]:
# group_cols = ['task_name', 'model', 'mem', 'segment', 'learning_rate']
# group_cols = ['task_name', 'model', 'mem', 'learning_rate', 'modification']
group_cols = ['n_pairs', 'model', 'mem', 'learning_rate', 'modification']
# group_cols = ['task_name', 'model', 'mem']

grouped = df.groupby(group_cols, dropna=False).agg(
    avg_em=('eval_exact_match', lambda x: round(np.nanmean(x) * 100, 2)),
    std_em=('eval_exact_match', lambda x: round(np.nanstd(x) * 100, 2)),
    all_ems=('eval_exact_match', lambda x: tuple(round(v * 100, 2) for v in x if not np.isnan(v))),
    n_runs=('eval_exact_match', 'count'),
)

# grouped.sort_values(['task_name', 'avg_em'], ascending=[True, False])

In [14]:
grouped

avg_em  \
n_pairs model                       mem learning_rate modification           
1       flashrnn_elman              0   0.000300      2x steps      100.00   
                                        0.001000      2x steps      100.00   
        flashrnn_gru                0   0.000300      2x steps      100.00   
                                        0.001000      2x steps      100.00   
        flashrnn_lstm               0   0.000300      2x steps      100.00   
                                        0.001000      2x steps      100.00   
        flashrnn_slstm              0   0.000300      2x steps      100.00   
                                        0.001000      2x steps      100.00   
        gated_delta_net             0   0.000300                    100.00   
                                        0.001000                    100.00   
        rmca-v1_llama               8   0.000300                    100.00   
                                        0.001000                     99.82   
        rmca_llama                  8   0.000300                    100.00   
                                        0.001000                    100.00   
2       flashrnn_elman              0   0.000300      2x steps       48.42   
                                        0.001000      2x steps       50.80   
        flashrnn_gru                0   0.000300      2x steps       51.82   
                                        0.001000      2x steps       51.22   
        flashrnn_lstm               0   0.000300      2x steps       50.68   
                                        0.001000      2x steps       51.30   
        flashrnn_slstm              0   0.000300      2x steps       99.14   
                                        0.001000      2x steps       51.84   
        rmca-v1.1_llama             8   0.001000                     94.74   
        rmca_llama                  8   0.000300                     98.98   
                                        0.001000                     94.84   
4       delta_net                   0   0.000100                     99.97   
                                        0.000300                     99.94   
                                        0.001000                     99.95   
                                        0.003000                     99.96   
        flashrnn_elman              0   0.000300      2x steps        7.82   
                                        0.001000      2x steps        5.72   
        flashrnn_gru                0   0.000300      2x steps       24.96   
                                        0.001000      2x steps       23.68   
        flashrnn_lstm               0   0.000300      2x steps       22.62   
                                        0.001000      2x steps       23.26   
        flashrnn_slstm              0   0.000300      2x steps       84.42   
                                        0.001000      2x steps       25.92   
        gated_delta_net             0   0.000100                    100.00   
                                        0.000300                    100.00   
                                        0.001000                     99.99   
                                        0.003000                     99.96   
        linear_attention            0   0.000100                      0.01   
                                        0.000300                      0.02   
                                        0.001000                      0.07   
                                        0.003000                      0.05   
        mamba2                      0   0.000300                    100.00   
                                        0.001000                     99.96   
        rmca-v1.1_llama             4   0.000300                     25.50   
                                    8   0.000300                     25.74   
                                        0.001000                     25.84   
            

In [11]:
# Select only the best row for each (task_name, model), corresponding to the best memory size, learning rate, and modification, ranked by avg_em
best_rows = grouped.reset_index().sort_values(['n_pairs', 'avg_em'], ascending=[True, False]).groupby(['n_pairs', 'model'], as_index=False).head(1)
# best_rows

In [12]:
best_rows[best_rows.n_pairs == 16]

,n_pairs,model,mem,learning_rate,modification,avg_em,std_em,all_ems,n_runs
161,16,gated_delta_net,0,0.0030,,99.97,0.01,"(99.98, 99.96, 99.98)",3
167,16,mamba2,0,0.0010,,99.92,0.00,"(99.92,)",1
155,16,delta_net,0,0.0003,,99.80,0.03,"(99.8, 99.84, 99.76)",3
174,16,rwkv6,0,0.0003,,86.17,4.42,"(88.3, 90.2, 80.02)",3
171,16,rmca-v2.0-write-order_llama,32,0.0001,2x steps,30.78,34.04,"(78.92, 6.66, 6.76)",3
168,16,rmca-v1.1_llama,8,0.0010,,0.50,0.00,"(0.5,)",1
164,16,linear_attention,0,0.0010,,0.04,0.03,"(0.02, 0.02, 0.08)",3


In [13]:
# group_cols = ['task_name', 'model', 'mem', 'segment', 'learning_rate']
group_cols = ['task_name', 'model', 'mem', 'learning_rate', 'modification']

grouped = df.groupby(group_cols, dropna=False).agg(
    avg_em=('eval_exact_match', lambda x: round(np.nanmean(x) * 100, 2)),
    std_em=('eval_exact_match', lambda x: round(np.nanstd(x) * 100, 2)),
    all_ems=('eval_exact_match', lambda x: tuple(round(v * 100, 2) for v in x if not np.isnan(v))),
    n_runs=('eval_exact_match', 'count'),
)

grouped.sort_values(['task_name', 'avg_em'], ascending=[True, False])

avg_em  \
task_name       model                       mem learning_rate modification           
N1-K2V2-V62_1M  flashrnn_elman              0   0.000300      2x steps      100.00   
                                                0.001000      2x steps      100.00   
                flashrnn_gru                0   0.000300      2x steps      100.00   
                                                0.001000      2x steps      100.00   
                flashrnn_lstm               0   0.000300      2x steps      100.00   
                                                0.001000      2x steps      100.00   
                flashrnn_slstm              0   0.000300      2x steps      100.00   
                                                0.001000      2x steps      100.00   
                gated_delta_net             0   0.000300                    100.00   
                                                0.001000                    100.00   
                rmca-v1_llama               8   0.000300                    100.00   
                rmca_llama                  8   0.000300                    100.00   
                                                0.001000                    100.00   
                rmca-v1_llama               8   0.001000                     99.82   
N16-K2V2-V62_1M gated_delta_net             0   0.003000                     99.97   
                                                0.000100                     99.95   
                                                0.000300                     99.94   
                                                0.001000                     99.93   
                mamba2                      0   0.001000                     99.92   
                                                0.000300                     99.84   
                delta_net                   0   0.000300                     99.80   
                                                0.000100                     99.69   
                                                0.001000                     99.61   
                                                0.003000                     99.33   
                rwkv6                       0   0.000300                     86.17   
                                                0.000100                     50.05   
                rmca-v2.0-write-order_llama 32  0.000100      2x steps       30.78   
                                                0.000300      2x steps        3.65   
                                            8   0.000100      2x steps        2.50   
                                                0.000300      2x steps        0.54   
                rmca-v1.1_llama             8   0.001000                      0.50   
                rwkv6                       0   0.001000                      0.06   
                linear_attention            0   0.001000                      0.04   
                rwkv6                       0   0.003000                      0.04   
                linear_attention            0   0.000100                      0.01   
                                                0.000300                      0.01   
                                                0.003000                      0.01   
N2-K2V2-V62_1M  flashrnn_slstm              0   0.000300      2x steps       99.14   
                rmca_llama                  8   0.000300                     98.98   
                                                0.001000                     94.84   
                rmca-v1.1_llama             8   0.001000                     94.74   
                flashrnn_slstm              0   0.001000      2x steps       51.84   
                flashrnn_gru                0   0.000300      2x steps       51.82   
                flashrnn_lstm               0   0.001000      2x steps       51.30   
                flashrnn_gru                0   0.001000      2x steps       51.22   
                flashrnn_elman    

In [15]:
df['model']

,model,model
0,rmca_llama,rmca_llama
1,flashrnn_elman,flashrnn_elman
2,rmca-v1_llama,rmca-v1_llama
3,flashrnn_slstm,flashrnn_slstm
4,rmca-v1_llama,rmca-v1_llama
...,...,...
376,rmca-v2.0-write-order_llama,rmca-v2.0-write-order_llama
377,rmca-v2.0-write-order_llama,rmca-v2.0-write-order_llama
378,rmca-v2.0-write-order_llama,rmca-v2.0-write-order_llama
379,rmca-v2.0-write-order_llama,rmca-v2.0-write-order_llama


In [ ]:
group_cols = ['task_name', 'model', 'mem', 'segment', 'learning_rate']

grouped = df.groupby(group_cols, dropna=False).agg(
    avg_em=('eval_exact_match', lambda x: round(np.nanmean(x) * 100, 2)),
    std_em=('eval_exact_match', lambda x: round(np.nanstd(x) * 100, 2)),
    all_ems=('eval_exact_match', lambda x: tuple(round(v * 100, 2) for v in x if not np.isnan(v))),
    n_runs=('eval_exact_match', 'count'),
).reset_index()

pivot = grouped.pivot_table(
    index=['model', 'mem', 'segment', 'learning_rate'],
    columns='task_name',
    values='avg_em',
)
pivot.sort_index()

task_name                                               N1-K2V2-V62_1M  \
model                       mem  segment learning_rate                   
rmca-v1.1_llama             4.0  1x4     0.000300                  NaN   
                            8.0  1x16    0.001000                  NaN   
                                 1x2     0.001000                  NaN   
                                 1x4     0.000300                  NaN   
                                         0.001000                  NaN   
                                 1x8     0.001000                  NaN   
                            32.0 1x4     0.000300                  NaN   
rmca-v1_llama               8.0  1x1     0.000300               100.00   
                                         0.001000                99.82   
rmca-v2.0-write-order_llama 1.0  1x4     0.000300                  NaN   
                                 1x8     0.000300                  NaN   
                            2.0  1x4     0.000300                  NaN   
                                 1x8     0.000300                  NaN   
                            4.0  1x4     0.000300                  NaN   
                                 1x8     0.000300                  NaN   
                            8.0  1x4     0.000001                  NaN   
                                         0.000003                  NaN   
                                         0.000010                  NaN   
                                         0.000030                  NaN   
                                         0.000100                  NaN   
                                         0.000300                  NaN   
                                 1x8     0.000001                  NaN   
                                         0.000003                  NaN   
                                         0.000010                  NaN   
                                         0.000030                  NaN   
                                         0.000100                  NaN   
                                         0.000300                  NaN   
                                         0.001000                  NaN   
                                         0.003000                  NaN   
                            16.0 1x4     0.000300                  NaN   
                                 1x8     0.000300                  NaN   
                            32.0 1x4     0.000300                  NaN   
                                 1x8     0.000300                  NaN   
rmca_llama                  1.0  1x4     0.000300                  NaN   
                                         0.001000                  NaN   
                                 8x1     0.000300                  NaN   
                                         0.001000                  NaN   
                            2.0  1x4     0.000300                  NaN   
                                         0.001000                  NaN   
                                 8x1     0.000300                  NaN   
                            8.0  1x1     0.000300               100.00   
                                         0.001000               100.00   
                                 1x2     0.000300                  NaN   
                                         0.001000                  NaN   
                                 1x4     0.000300                  NaN   
                                         0.001000                  NaN   
                                 1x8     0.000300                  NaN   
                                         0.001000                  NaN   
                                 8x1     0.000300                  NaN   
                                         0.001000                  NaN   
                            16.0 1x4     0.000300                  NaN   
                                         0.001000                  NaN   
    

In [ ]:
# # group_cols = ['model', 'mem', 'task_name', 'n_pairs', 'learning_rate']
# group_cols = ['task_name', 'model', 'mem', 'learning_rate']

# grouped = df.groupby(group_cols, dropna=False).agg(
#     avg_em=('eval_exact_match', lambda x: round(np.nanmean(x) * 100, 2)),
#     std_em=('eval_exact_match', lambda x: round(np.nanstd(x) * 100, 2)),
#     all_ems=('eval_exact_match', lambda x: tuple(round(v * 100, 2) for v in x if not np.isnan(v))),
#     n_runs=('eval_exact_match', 'count'),
# )

# grouped.sort_values(['model', 'mem', 'avg_em'], ascending=[True, True, False])

avg_em  \
task_name       model                       mem  learning_rate           
N4-K2V2-V62_1M  delta_net                   0.0  0.000100        99.97   
                                                 0.003000        99.96   
                                                 0.001000        99.95   
N8-K2V2-V62_1M  delta_net                   0.0  0.000100        99.95   
N4-K2V2-V62_1M  delta_net                   0.0  0.000300        99.94   
N8-K2V2-V62_1M  delta_net                   0.0  0.000300        99.94   
                                                 0.003000        99.86   
                                                 0.001000        99.85   
N16-K2V2-V62_1M delta_net                   0.0  0.000300        99.80   
                                                 0.000100        99.69   
                                                 0.001000        99.61   
                                                 0.003000        99.33   
N32-K2V2-V62_1M delta_net                   0.0  0.000300        99.31   
                                                 0.001000        98.39   
                                                 0.000100        98.12   
                                                 0.003000        98.11   
N1-K2V2-V62_1M  flashrnn_elman              0.0  0.000300       100.00   
                                                 0.001000       100.00   
N2-K2V2-V62_1M  flashrnn_elman              0.0  0.001000        50.80   
                                                 0.000300        48.42   
N4-K2V2-V62_1M  flashrnn_elman              0.0  0.000300         7.82   
                                                 0.001000         5.72   
N8-K2V2-V62_1M  flashrnn_elman              0.0  0.000300         2.74   
                                                 0.001000         1.61   
N1-K2V2-V62_1M  flashrnn_gru                0.0  0.000300       100.00   
                                                 0.001000       100.00   
N2-K2V2-V62_1M  flashrnn_gru                0.0  0.000300        51.82   
                                                 0.001000        51.22   
N4-K2V2-V62_1M  flashrnn_gru                0.0  0.000300        24.96   
                                                 0.001000        23.68   
N8-K2V2-V62_1M  flashrnn_gru                0.0  0.001000         7.74   
                                                 0.000300         2.94   
N1-K2V2-V62_1M  flashrnn_lstm               0.0  0.000300       100.00   
                                                 0.001000       100.00   
N2-K2V2-V62_1M  flashrnn_lstm               0.0  0.001000        51.30   
                                                 0.000300        50.68   
N4-K2V2-V62_1M  flashrnn_lstm               0.0  0.001000        23.26   
                                                 0.000300        22.62   
N8-K2V2-V62_1M  flashrnn_lstm               0.0  0.001000         9.90   
                                                 0.000300         3.70   
                                                 0.010000         0.02   
N1-K2V2-V62_1M  flashrnn_slstm              0.0  0.000300       100.00   
                                                 0.001000       100.00   
N2-K2V2-V62_1M  flashrnn_slstm              0.0  0.000300        99.14   
N4-K2V2-V62_1M  flashrnn_slstm              0.0  0.000300        84.42   
N2-K2V2-V62_1M  flashrnn_slstm              0.0  0.001000        51.84   
N8-K2V2-V62_1M  flashrnn_slstm              0.0  0.001000        32.91   
N4-K2V2-V62_1M  flashrnn_slstm              0.0  0.001000        25.92   
N8-K2V2-V62_1M  flashrnn_slstm              0.0  0.000300        14.44   
N1-K2V2-V62_1M  gated_delta_net             0.0  0.000300       100.00   
                                                 0.001000       100.00   
N4-K2V2-V62_1M  gated_delta_net             0.0  0.000100       100.00   
                                                 0.000300       100.